In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy import stats
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_curve, auc
from sklearn.preprocessing import label_binarize
import statsmodels.formula.api as smf
from statsmodels.stats.contingency_tables import mcnemar
from numpy.linalg import LinAlgError
import warnings

# Load data

In [2]:
with warnings.catch_warnings(): # Excel Header cannot be parsed
    warnings.simplefilter("ignore")
    data = pd.read_excel('./SAh_OKIE_DB_20250303.xlsx', sheet_name=0, header=0, engine='openpyxl')
    med = pd.read_excel('./SAh_OKIE_DB_meds_20250217.xlsx', sheet_name=0, header=0, engine='openpyxl')
    coc = pd.read_csv('./coc.csv')

# Preprocessing

### Primary Outcome
COC Decision, Multiclass

In [3]:
coc_mapping = {
    0: 0, # Back Home
    1: 3, # Nursing Home
    2: 3, # Nursing Home
    3: 2, # Rehabilitation
    4: 2, # Rehabilitation
    5: 1, # Acute Geriatric Clinic
    6: 4 # Other Acute Clinic
}

In [4]:
data['coc_gt'] = data['dis_geriatrician_0'].map(coc_mapping)
data['coc_soc'] = data['discharge_general'].map(coc_mapping)

#### COC congruent (Y/N)
binary, 1 = Yes, 0 = No

In [5]:
def map_coc_congruent(row):
    if pd.isna(row['coc_gt']):
        return np.nan # If ground truth is missing
    else:
        return row['coc_gt'] == row['coc_soc']

In [6]:
data['coc_congruent'] = data.apply(map_coc_congruent, axis=1)

### Secondary Outcomes

In [7]:
def binarize_endpoint(row, col_pre, col_post, inverse=False):
    if pd.isna(row[col_pre]) or pd.isna(row[col_post]): # if either are missing
        return np.nan
    if row[col_pre] == row[col_post]: # If outcome score stayed the same
        return 0
    if row[col_pre] < row[col_post]: # If outcome score went up
        return 0 if inverse==False else 1
    if row[col_pre] > row[col_post]: # If outcome score went down
        return 1 if inverse==False else 0
    else: # Exception handling
        return np.nan

#### Barthel Index change
binary, 0 = better/same, 1 = worse

In [8]:
data['bi_decline_3mo'] = data.apply(
    binarize_endpoint, # better/same: 0, worse: 1
    col_pre='bi_prior',
    col_post='bi_followup',
    axis=1
)

In [9]:
data['bi_decline_15mo'] = data.apply(
    binarize_endpoint, # better/same: 0, worse: 1
    col_pre='bi_prior',
    col_post='bi_fu12',
    axis=1
)

#### CHARMI
Ordinal Scale: 0-10 \
Extra Category "wheelchair" encoded as 11

In [10]:
charmi_list = ['charmi_prior', 'charmi_postop_1', 'charmi_postop_3', 'charmi_postop_5', 'charmi_postop_7', 'charmi_followup']
data[charmi_list] = data[charmi_list].replace({11: np.nan}) # Replaces "wheelchair" with None.

In [11]:
data['charmi_fu12'] = data['charmi_fu12'].map(lambda x: np.nan if x == 'W' else x)
data['charmi_fu12'] = pd.to_numeric(data['charmi_fu12'])

#### CHARMI change
binary, 0 = better/same, 1 = worse

In [12]:
data['charmi_decline_3mo'] = data.apply(
    binarize_endpoint, # better/same: 0, worse: 1
    col_pre='charmi_prior',
    col_post='charmi_followup',
    axis=1
)

In [13]:
data['charmi_decline_15mo'] = data.apply(
    binarize_endpoint, # better/same: 0, worse: 1
    col_pre='charmi_prior',
    col_post='charmi_fu12',
    axis=1
)

#### New Mobility Scale (NMS) change
binary, 0 = better/same, 1 = worse

In [14]:
data['life_space_decline_3mo'] = data.apply(
    binarize_endpoint, # better/same: 0, worse: 1
    col_pre='life_space_prior',
    col_post='life_space_followup',
    axis=1
)

In [15]:
data['life_space_decline_15mo'] = data.apply(
    binarize_endpoint, # better/same: 0, worse: 1
    col_pre='life_space_prior',
    col_post='life_space_fu12',
    axis=1
)

#### Social Grade (Y/N)
binary, 1 = Yes, 0 = No

In [16]:
def map_social_grade(row, col):
    if row[col] == 98: # Social Grade was applied for, but the decision is pending.
        return 0
    if row[col] == 100: # Participant refused to answer
        return np.nan
    else:
        return row[col]

In [17]:
data['social_grade'] = data.apply(map_social_grade, col='social_grade', axis=1)

In [18]:
data['social_grade_followup'] = data.apply(map_social_grade, col='social_followup_grade', axis=1)

#### Social Grade level
Ordinal Scale: 0-5

In [19]:
def map_social_gradelevel(row, grade, gradelevel):
    if row[grade] == 0: # No social grade, or social grade was applied for, but the decision is pending.
        return 0
    if pd.isna(row[grade]): # Participant refused to answer, or missing value
        return np.nan
    else:
        return row[gradelevel]

In [20]:
data['social_gradelevel'] = data.apply(
    map_social_gradelevel,
    grade='social_grade',
    gradelevel='social_gradelevel',
    axis=1
)

In [21]:
data['social_gradelevel_followup'] = data.apply(
    map_social_gradelevel,
    grade='social_grade_followup',
    gradelevel='social_followup_gradelevel',
    axis=1
)

In [22]:
data['social_gradelevel_fu12'] = data.apply(
    map_social_gradelevel,
    grade='social_gradez',
    gradelevel='social_gradeb',
    axis=1
)

#### Social Grade level change
binary, 0 = lower/same, 1 = increase

In [23]:
data['social_gradelevel_increase_3mo'] = data.apply(
    binarize_endpoint, # lower/same: 0, increase: 1
    col_pre='social_gradelevel',
    col_post='social_gradelevel_followup',
    inverse=True,
    axis=1
)

In [24]:
data['social_gradelevel_increase_15mo'] = data.apply(
    binarize_endpoint, # lower/same: 0, increase: 1
    col_pre='social_gradelevel',
    col_post='social_gradelevel_fu12',
    inverse=True,
    axis=1
)

#### Nursing service
Ordinal Scale 0-3 \
0 = home without support, 1 = home with support, 2 = assisted living, 3 = nursing home

In [25]:
def map_nursing_service(row, accomodation, help):
    # Mapping with decreasing need for nursing services
    if row[accomodation] <=1 and row[help] == 0:
        return 0 # Home without support
    if row[accomodation] <=1 and row[help] == 1:
        return 1 # Home with support
    if row[accomodation] == 2:
        return 2 # assisted living
    if row[accomodation] == 4:
        return 3 # nursing home
    else: # inconclusive or missing data
        return np.nan

In [26]:
data['nursing_service'] = data.apply(
    map_nursing_service,
    accomodation = 'social_accomodation',
    help = 'social_help',
    axis=1
)

In [27]:
data['nursing_service_followup'] = data.apply(
    map_nursing_service,
    accomodation = 'social_followup_accomodation',
    help = 'social_followup_help',
    axis=1
)

#### Nursing service change
binominal scale 0 = lower/same, 1 = increase

In [28]:
data['nursing_service_increase_3mo'] = data.apply(
    binarize_endpoint, # lower/same: 1, increase: 0
    col_pre='nursing_service',
    col_post='nursing_service_followup',
    inverse=True,
    axis=1
)

#### Institutionalization (Y/N)
binary, 1 = Yes, 0 = No

In [29]:
def map_institution(row, col):
    if row[col] == 4:
        return 1 # Living in institution
    if row[col] in (0, 1, 2, 3, 98):
        return 0 # Not living in institution
    else: # 100 (denies answer), 101 (cannot answer)
        return np.nan

In [30]:
data['institution'] = data.apply(map_institution, col='social_accomodation', axis=1)

In [31]:
data['institution_followup'] = data.apply(map_institution, col='social_followup_accomodation', axis=1)

In [32]:
# correcting wrong data
data.loc[data['id']==11127, 'institution_followup'] = 1
data.loc[data['id']==12008, 'institution'] = 0

In [33]:
def map_institution_fu12(row):
    if row['social_livingz'] in (1, 2):
        return 1 # Living in institution
    if row['social_livingz'] in (3, 4, 5, 6, 7, 8, 9):
        return 0 # Not living in institution
    else:
        return np.nan

In [34]:
data['institution_fu12'] = data.apply(map_institution_fu12, axis=1)

#### Institutionalization change (Y/N)
binary, 1 = Yes, 0 = No/same

In [35]:
def map_institution_change(row, col):
    if pd.isna(row[col]) or pd.isna(row['institution']):
        return np.nan
    if row[col] == row['institution']:
        return 0
    if (row[col] == 0) and (row['institution'] == 1):
        return 0
    if (row[col] == 1) and (row['institution'] == 0):
        return 1
    else:
        return np.nan

In [36]:
data['institution_change_3mo'] = data.apply(map_institution_change, col='institution_followup', axis=1)

In [37]:
data['institution_change_15mo'] = data.apply(map_institution_change, col='institution_fu12', axis=1)

In [38]:
data.loc[
    (data['institution_change_15mo'] == True) | (data['institution_change_3mo'] == True),
    ['id', 'institution', 'institution_followup', 'institution_fu12', 'coc_congruent']
]

,id,institution,institution_followup,institution_fu12,coc_congruent
90,11096,0.0,1.0,NaN,True
124,11133,0.0,1.0,1.0,True
128,11137,0.0,1.0,0.0,True
138,12008,0.0,0.0,1.0,True


#### Readmission (Y/N)
within 2d or 30d combined

In [39]:
def map_readmission(row):
    if row['d_c_wiedervor2t_trigger'] == 'true' or row['d_c_wiederauf90t_trigger'] == 'true':
        return 1  # Readmitted to UKU within 3mo
    if row['d_c_wiedervor2t_trigger'] == 'false' and row['d_c_wiederauf90t_trigger'] == 'false':
        return 0  # No readmission to UKU within 3mo
    else:
        return np.nan

In [40]:
data['readmission_3mo'] = data.apply(map_readmission, axis=1)

### Explanatory Variables

#### Surgery
one-hot-encodings for UCH, AVC and URO \
binary, 1 = Yes, 0 = No

In [41]:
data = pd.get_dummies(data, columns=['station'], prefix='', prefix_sep='')

#### Living alone (Y/N)
binary, 1 = Yes, 0 = No

In [42]:
def map_living_alone(row, col):
    if row[col] == '0':
        return 1 # Living alone
    if row[col] in ('1', '2', '3', '5', '1,2', '1,2,3', '1,3', '98'):
        return 0 # Not living alone
    else:
        return np.nan

In [43]:
data['living_alone'] = data.apply(map_living_alone, col='social_living',  axis=1)

In [44]:
data['living_alone_followup'] = data.apply(map_living_alone, col='social_followup_living', axis=1)

#### Number of Medications
Discrete variable (n)

In [45]:
number_of_medications_dict={}
med = med[med['slot'].str.contains('preop')] # Only pre-operative medication
med = med[med['freq_type'] != 'bb'] # drop on-demand medication
for ID in data['id']:
    number_of_medications_dict[ID] = med[med['ID'] == ID].shape[0]
data['number_of_medications'] = data['id'].map(number_of_medications_dict)

#### ASA Class
ordinal variable 1-5

In [46]:
data['comorbidity_1_asa'] = data['comorbidity_1_asa'] + 1

#### Dementia (Y/N)
binary, 1 = Yes, 0 = No

In [47]:
dementia_anamnesis = (data['comorbidity_2_dementia'] > 0) & (data['comorbidity_2_dementia'] < 99) # yes = 1, 2; no = 0, 99 
dementia_anasthesia = (data['comorbidity_1_dementia'] == 1)
data['dementia'] = 0
data.loc[dementia_anamnesis | dementia_anasthesia, 'dementia'] = 1 # Either anamnesis or anaesthesia say dementia is present

#### Time to Surgery
Continous variable (hour)

In [48]:
data['time_to_operation'] = data['time_to_operation']/60 # in hours

#### Length of Stay on ICU
Continuous variable (min)

In [49]:
data.loc[data['los_icu'] == " ", 'los_icu'] = np.nan
data['los_icu'] = pd.to_numeric(data['los_icu'].str.replace(',', '.')) # in hours
data['los_icu'] = data['los_icu'] * 60 # in minutes

#### Falls (Y/N)
within last 3 months \
binary, 1 = Yes, 0 = No

In [50]:
data['falls'] = data['falls_1']
data.loc[data['falls'].isin([99, 100, 101]) , 'falls'] = np.nan
data.loc[data['falls'] > 0 , 'falls'] = 1

In [51]:
data['falls_followup'] = data['medical_history_2']
data.loc[data['falls_followup'].isin([99, 100, 101]) , 'falls_followup'] = np.nan
data.loc[data['falls_followup'] > 0 , 'falls_followup'] = 1

### Combine Datasets

In [52]:
column_mapping = {
    'adm_age': 'Alter bei OP (Jahre)',
    'isar': 'ISAR (Score, ordinal)',
    'bi_preop': 'Barthel-Index bei Aufnahme (Score, ordinal)',
    'cfs_prior': 'Clinical Frailty Scale (Score, ordinal)',
    'cut_to_suture': 'OP Dauer (Minuten)',
}

In [53]:
data.loc[data['id'] == 11005, 'cut_to_suture'] = 80 # add missing data

In [54]:
def get_id(row, cols):
    if row[cols].isna().any():
        return None
    row['identifier'] = ''
    for col in cols:
        row['identifier'] = row['identifier'] + '_' + str(int(row[col]))
    return row['identifier']

In [55]:
coc['identifier'] = coc.apply(get_id, cols=column_mapping.values(), axis=1)
data['identifier'] = data.apply(get_id, cols=column_mapping.keys(), axis=1)

In [56]:
data = pd.merge(data, coc, on='identifier', how='inner')
set(coc['identifier'].values) - set(data['identifier'].values)

{'_72_3_100_5_146', '_82_2_100_2_162'}

#### COC Prediction Congruent (Y/N)
binary, 1 = Yes, 0 = No

In [57]:
def map_coc_pred_congruent(row):
    if pd.isna(row['coc_gt']):
        return np.nan # If ground truth is missing
    else:
        return row['coc_gt'] == row['coc_pred']

In [58]:
data['coc_pred_congruent'] = data.apply(map_coc_pred_congruent, axis=1).astype(int)

### Drop-Outs

In [59]:
data[data['coc_congruent'].isna()][['id', 'dropout', 'dropout_reason']]

,id,dropout,dropout_reason


# Select Variables

In [60]:
variables = [
    # Primary Outcome
    'coc_gt', # COC ground truth
    'coc_soc', # COC standard of care
    'coc_congruent', # Is COC suggestion and real discharge destination equivalent? Y/N
    'coc_pred', # COC prediction from the COC algorithm
    'coc_pred_congruent', # Is COC prediction and destination suggested by expert equivalent? Y/N
    
    # Secondary Outcomes (binarized)
    # 3 months follow-up
    'social_gradelevel_increase_3mo', # Change Care level, 0: lower/same, 1: increase
    'nursing_service_increase_3mo', # Change Nursing service, 0: lower/same, 1: increase
    'bi_decline_3mo', # Change BI, 0: better/same, 1: decline
    'charmi_decline_3mo', # Change CHARMI, 0: better/same, 1: decline
    'life_space_decline_3mo', # Change NMS, 0: better/same, 1: decline
    'readmission_3mo', # Readmission within 3mo
    'institution_change_3mo', # Insitutionalization, 0: No/same, 1: new institutionalization
    # 15 months follow-up
    'social_gradelevel_increase_15mo', # Change Care level, 0: lower/same, 1: increase
    'bi_decline_15mo', # Change BI, 0: better/same, 1: decline
    'charmi_decline_15mo', # Change CHARMI, 0: better/same, 1: decline
    'life_space_decline_15mo', # Change NMS, 0: better/same, 1: decline
    'institution_change_15mo', # Insitutionalization, 0: No/same, 1: new institutionalization

    # Group Variables
    'UCH', # Trauma Surgery
    'AVC', # General & Visceral Surgery
    'URO', # Urology
        
    # Explanatory Variables
    'adm_age', # Age at the time of operation
    'sex', # w:0 m:1
    'moca_preop', # Cognition MoCA 5min preop
    'phq4_preop', # PHQ4 preop
    'phq4_followup', # PHQ4 3 months
    'cfs_prior', # CFS preop (nurse)
    'nursing_service', # 0 = no support, 1 = support at home, 2 = assisted living, 3 = nursing home
    'living_alone', # Living alone (Y/N) prior
    'bmi_preop', # Nutrition (BMI)
    'malnutrition', # Nutrition Risk Screening (NRS) preop
    'number_of_medications', # Number of medications preop
    'comorbidity_1_asa', # ASA
    'isar', # ISAR
    'falls', # Has fallen in the last 3 months preop # CL: Too much bias, as nearly all Trauma patients have fallen?
    'los_days', # length of stay in days
    'time_to_operation', # Time-to-surgery in hours
    'cut_to_suture', # Cut-to-suture time in min
    'los_icu', # Length of stay on ICU in min
    'op_type', # 0:elective, 1:emergency
    'dementia', # Dementia in Anamnesis or According to Anaesthesist
    'bi_prior', # Barthel index prior
    'life_space_prior', # New mobility score preop
    'charmi_prior', # CHARMI prior
    'social_gradelevel', # Care level X/5 on admission
]

In [61]:
df = data[variables].copy()
df = df.dropna(subset=['coc_pred_congruent'])
# df.to_csv('./OKIE_data.csv', index=False)

# Analysis

In [62]:
explanatory_variables = {
    'adm_age': 'continuous',
    'sex': 'binary',
    'UCH': 'binary',
    'AVC': 'binary',
    'URO': 'binary',
    'bmi_preop': 'continuous',
    'malnutrition': 'binary',
    'moca_preop': 'ordinal',
    'dementia': 'binary',
    'phq4_preop': 'ordinal',
    'isar': 'ordinal',
    'cfs_prior': 'ordinal',
    'comorbidity_1_asa': 'ordinal',
    'number_of_medications': 'discrete',
    'social_gradelevel': 'ordinal',
    'nursing_service': 'ordinal',
    'living_alone': 'binary',
    'bi_prior': 'ordinal',
    'charmi_prior': 'ordinal',
    'life_space_prior': 'ordinal',
    'falls': 'binary',
    'op_type': 'binary',
    'time_to_operation': 'continuous',
    'cut_to_suture': 'continuous',
    'los_icu': 'continuous',
    'los_days': 'discrete',
}

explanatory_variable_names = [
    'Age (years)',
    'Sex (male)',
    'Trauma Surgery (Yes/No)',
    'General & Visceral Surgery (Yes/No)',
    'Urology (Yes/No)',
    'BMI (kg/m²)',
    'NRS (Yes/No)',
    'MoCA 5-min (score)',
    'Dementia (Yes/No)',
    'PHQ4 (score)',
    'ISAR (score)',
    'CFS (score)',
    'ASA (class)',
    'Number of Medications (n)',
    'Social Grade Level (class)',
    'Nursing Services (class)',
    'Living Alone (Yes/No)',
    'Barthel Index (score)',
    'CHARMI (score)',
    'NMS (score)',
    'Falls (Yes/No)',
    'Emergency OP (Yes/No)',
    'Time to OP (hours)',
    'Cut-to-Suture Time (minutes)',
    'Length of Stay ICU (minutes)',
    'Length of Stay (days)',
    ]

outcomes = {
    # 3 months follow-up
    'Care Grade Level (3 months)': 'social_gradelevel_increase_3mo',
    'Nursing Service (3 months)': 'nursing_service_increase_3mo',
    'Barthel Index (3 months)': 'bi_decline_3mo',
    'CHARMI (3 months)': 'charmi_decline_3mo',
    'New Mobility Scale (3 months)': 'life_space_decline_3mo',
    'Readmission (3 months)': 'readmission_3mo',
    'Institutionalization (3 months)': 'institution_change_3mo',
    # 12 months follow-up
    'Care Grade Level (12 months)': 'social_gradelevel_increase_12mo',
    'Barthel Index (12 months)': 'bi_decline_12mo',
    'CHARMI (12 months)': 'charmi_decline_12mo',
    'New Mobility Scale (12 months)': 'life_space_decline_12mo',
    'Institutionalization (12 months)': 'institution_change_12mo',
}

## Primary Endpoint

### Functions

In [63]:
def select_top_rows(y_proba, data_array, percentile):
    
    """
    Selects rows from a data array based on a probability threshold.

    This function identifies and returns rows from a data array corresponding to samples with high predicted probabilities.
    It calculates a threshold based on the specified percentile of the maximum predicted probabilities for each sample and
    then selects the rows where the maximum predicted probability exceeds this threshold.

    Args:
        y_proba (np.ndarray): Predicted probabilities for each sample (shape: (n_samples, n_classes)).
        data_array (np.ndarray): The data array from which to select rows.
        percentile (float): The percentile to use for calculating the probability threshold.

    Returns:
        np.ndarray: A subset of the data array containing only the rows with high predicted probabilities.
    """
    
    threshold = np.percentile(np.max(y_proba, axis=1), percentile)
    indices = np.where(np.max(y_proba, axis=1) >= threshold)[0]
    
    return data_array[indices]
    

def get_mcnemar(coc_gt, coc_soc, coc_pred):
    """
    Compare two prediction methods against the ground truth using McNemar's test.

    Parameters
    ----------
    coc_gt : array-like
        Ground truth labels (0/1 or False/True) for each sample.
    coc_soc : array-like
        Predictions from method 1 (same shape as coc_gt).
    coc_pred : array-like
        Predictions from method 2 (same shape as coc_gt).

    Returns
    -------
    dict
        Dictionary with:
        - 'b': Number of samples where method 1 is correct and method 2 is wrong
        - 'c': Number of samples where method 1 is wrong and method 2 is correct
        - 'p_value': McNemar test p-value
    """
    coc_gt = np.asarray(coc_gt)
    coc_soc = np.asarray(coc_soc)
    coc_pred = np.asarray(coc_pred)
    
    if not (coc_gt.shape == coc_soc.shape == coc_pred.shape):
        raise ValueError("All input arrays must have the same shape.")
    
    # Determine correctness of each method
    soc_correct = coc_soc == coc_gt
    pred_correct = coc_pred == coc_gt
    
    # Build 2x2 contingency table
    # b = method 1 correct, method 2 wrong
    # c = method 1 wrong, method 2 correct
    b = np.sum(soc_correct & ~pred_correct)
    c = np.sum(~soc_correct & pred_correct)
    
    table = [[0, b],
             [c, 0]]
    
    # McNemar test with continuity correction
    result = mcnemar(table, exact=False, correction=True)
    
    return {'b': b, 'c': c, 'p_value': result.pvalue}


### All Predictions

In [64]:
y_true = data['coc_gt'].values
y_true_bin = label_binarize(y_true, classes=[0, 1, 2, 3])
y_soc = data['coc_soc'].values
y_pred = data['coc_pred'].values
class_cols = [f'coc_proba_class{i}' for i in range(4)]
y_proba = data[class_cols].to_numpy()
ncnemar_results = get_mcnemar(y_true, y_soc, y_pred)
print(f"p-value: {ncnemar_results['p_value']:.4f}")

p-value: 0.0411


### Top 85% Predictions

In [65]:
top_y_true = select_top_rows(y_proba, y_true, 15)
top_y_soc = select_top_rows(y_proba, y_soc, 15)
top_y_pred = select_top_rows(y_proba, y_pred, 15)
top_y_true_bin = select_top_rows(y_proba, y_true_bin, 15)
top_y_proba = select_top_rows(y_proba, y_proba, 15)
ncnemar_results = get_mcnemar(top_y_true, top_y_soc, top_y_pred)
print(f"p-value: {ncnemar_results['p_value']:.4f}")

p-value: 0.0058


### Stratification

Undertreatment

In [66]:
def is_undertreated(row):
    if pd.isna(row['coc_gt']) or pd.isna(row['coc_pred']): # missing data
        return np.nan
    elif row['coc_gt'] == row['coc_pred']: # correct treatment
        return np.nan
    elif (row['coc_gt'] in [1, 2]) and (row['coc_pred'] in [0, 3]): # Acute Geriatric Clinic or Rehab was correct, but went (nursing) home
        return True
    elif (row['coc_pred'] in [1, 2]) and (row['coc_gt'] in [0, 3]): # (Nursing) Home was correct, but went Acute Geriatric Clinic or Rehab
        return False
    else:
        return np.nan

In [67]:
df['is_undertreated'] = df.apply(is_undertreated, axis=1)

In [68]:
df.loc[df['is_undertreated'] == True, ['coc_gt', 'coc_pred']]

,coc_gt,coc_pred
5,1.0,3
13,1.0,0
27,1.0,0
28,2.0,0
38,2.0,0
40,2.0,0
54,1.0,0
60,1.0,0
105,1.0,0
121,1.0,0


In [69]:
df.loc[df['is_undertreated'] == False, ['coc_gt', 'coc_pred']]

,coc_gt,coc_pred
6,0.0,1
15,0.0,1
18,0.0,1
45,3.0,1
48,0.0,1
88,0.0,1
133,3.0,1
142,0.0,1
148,0.0,1


Time to Surgery by Ward

In [70]:
def tts_by_ward(ward):
    subset = df[ward]
    return pd.Series({
        'Mean': subset['time_to_operation'].mean(),
        'Standard Deviation': subset['time_to_operation'].std()
    })

In [71]:
pd.DataFrame({
    'UCH': tts_by_ward(df['UCH']),
    'AVC': tts_by_ward(df['AVC']),
    'URO': tts_by_ward(df['URO'])
}).T

,Mean,Standard Deviation
UCH,67.647222,92.038103
AVC,69.530303,94.233725
URO,22.937037,9.382748


## Descriptive Statistics (Table 1)

#### Helper Functions

In [72]:
def is_normal(df, target, var):
    for group in get_groups(df, target, var):
        if len(group) < 3:
            return False
        shapiro_stat, shapiro_p = stats.shapiro(group)
        if shapiro_p <= 0.05:
            return False
    return True

def get_groups(df, target, var):
    df_clean = df.dropna(subset=[target, var])
    groups = {}
    for i in df_clean[target].unique():
        groups[i] = df_clean.loc[df_clean[target] == i, var]
    return [groups[i] for i in sorted(groups.keys())]

### Analysis

In [73]:
table1 = pd.DataFrame()

In [74]:
df['coc_pred_congruent'] = df['coc_pred_congruent'].astype(int) # bool not accepted in statsmodels
coc_pred_congruent = df[df['coc_pred_congruent']==1]
coc_pred_not_congruent = df[df['coc_pred_congruent']==0]

subgroups = {
    'ALL': df,
    'COC+': coc_pred_congruent,
    'COC-': coc_pred_not_congruent,
}

In [75]:
for subgroup, subgroup_data in subgroups.items():
    for key, scale in explanatory_variables.items():
        if scale in ['continuous', 'discrete', 'ordinal']:
            table1.loc[key, f'{subgroup} mean (± sd)'] = f'{np.round(subgroup_data[key].mean(), 2)} (±{np.round(subgroup_data[key].std(), 2)})'
        elif scale == 'binary':
            table1.loc[key, f'{subgroup} n (%)'] = f'{int(subgroup_data[key].sum())} ({np.round(subgroup_data[key].sum()/len(subgroup_data)*100, 1)}%)'
        else:
            print('Error')

In [76]:
# target = 'coc_pred_congruent'
confounders = []
for variable, scale in explanatory_variables.items():
    if scale in ['continuous', 'discrete']:
        if is_normal(df, 'coc_pred_congruent', variable):
            _, p_value = stats.ttest_ind(*get_groups(df, 'coc_pred_congruent', variable))
        else:
            _, p_value = stats.mannwhitneyu(*get_groups(df, 'coc_pred_congruent', variable))
    elif scale=='ordinal':
        _, p_value = stats.mannwhitneyu(*get_groups(df, 'coc_pred_congruent', variable))
    elif scale=='binary':
        crosstab = pd.crosstab(df[variable], df['coc_pred_congruent'])
        _, p_value, _, _ = stats.chi2_contingency(crosstab)
    else:
        print('Error')
    if p_value < 0.05:
        confounders.append(variable) # Add explanatory variables to confounders list, if statistically significant
    table1.loc[variable, 'p_value'] = np.round(p_value,4)

### Table 1

In [77]:
table1 = table1.fillna('')
table1.index = explanatory_variable_names

In [78]:
table1

,ALL mean (± sd),ALL n (%),COC+ mean (± sd),COC+ n (%),COC- mean (± sd),COC- n (%),p_value
Age (years),80.51 (±6.29),,80.61 (±6.5),,80.1 (±5.27),,0.6520
Sex (male),,68 (40.7%),,58 (42.3%),,10 (33.3%),0.4815
Trauma Surgery (Yes/No),,127 (76.0%),,104 (75.9%),,23 (76.7%),1.0000
General & Visceral Surgery (Yes/No),,22 (13.2%),,17 (12.4%),,5 (16.7%),0.7440
Urology (Yes/No),,18 (10.8%),,16 (11.7%),,2 (6.7%),0.6335
BMI (kg/m²),26.28 (±5.03),,26.29 (±4.85),,26.23 (±5.93),,0.7000
NRS (Yes/No),,78 (46.7%),,66 (48.2%),,12 (40.0%),0.6204
MoCA 5-min (score),21.61 (±5.88),,21.63 (±5.78),,21.52 (±6.43),,0.9077
Dementia (Yes/No),,19 (11.4%),,15 (10.9%),,4 (13.3%),0.9560
PHQ4 (score),2.48 (±2.71),,2.27 (±2.56),,3.45 (±3.18),,0.0548


In [79]:
table1.to_csv('./table1.csv')